<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=343668913" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD, NEW SESSION: environment through Stage 11, EYEPACS, CUSTOM CNN + EFFICIENTNETB0 =====
# This closes out EyePACS's remaining two architectures. MobileNetV2 and ResNet50 are
# already done on this source. Once this completes, EyePACS is fully closed (4/4).

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

# ---------- CONFIG ----------
EYEPACS_CSV   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/labels/trainLabels15.csv'
EYEPACS_IMG   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/resized train 15'

GRADES      = ['0','1','2','3','4']
IMG_SIZE, BATCH_SIZE = 224, 32
SUBSAMPLE_SEED = 42
EYEPACS_TARGET = 3662
CUSTOM_LR = 1e-3
PHASE1_EPOCHS, PHASE1_LR, PHASE2_LR, EARLYSTOP_PAT, MONITOR = 10, 1e-3, 1e-5, 7, 'val_accuracy'
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)

# ---------- DATA REBUILD, EYEPACS ONLY ----------
eyepacs = pd.read_csv(EYEPACS_CSV)
eyepacs['grade']      = eyepacs['level'].astype(int).astype(str)
eyepacs['image_path'] = EYEPACS_IMG + '/' + eyepacs['image'].astype(str) + '.jpg'
eyepacs['source']     = 'eyepacs'
eyepacs['patient_id'] = eyepacs['image'].str.extract(r'^(\d+)_')
assert eyepacs['patient_id'].isna().sum() == 0, "EyePACS patient_id extraction failed"

def subsample_eyepacs(df, target_n=EYEPACS_TARGET, seed=SUBSAMPLE_SEED):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    frac = target_n / len(df)
    keep, _ = train_test_split(pg, train_size=frac, stratify=pg['grade'], random_state=seed)
    return df[df['patient_id'].isin(keep['patient_id'])].reset_index(drop=True)

eyepacs_s = subsample_eyepacs(eyepacs)
print(f"EyePACS subsampled: {len(eyepacs_s)} images, {eyepacs_s['patient_id'].nunique()} patients")

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, falling back to unstratified.")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_patient_level(df, rs=SEED, tag=""):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    pick = lambda ids: df[df['patient_id'].isin(ids['patient_id'])]
    tr, va, te = pick(p_tr), pick(p_va), pick(p_te)
    s_tr, s_va, s_te = set(p_tr['patient_id']), set(p_va['patient_id']), set(p_te['patient_id'])
    assert s_tr.isdisjoint(s_va) and s_tr.isdisjoint(s_te) and s_va.isdisjoint(s_te), f"{tag} PATIENT LEAKAGE"
    print(f"{tag} patient-leakage check: PASS")
    return tr, va, te

e_tr, e_va, e_te = split_patient_level(eyepacs_s, tag="EyePACS")

cls = np.array(GRADES)
cw = compute_class_weight('balanced', classes=cls, y=e_tr['grade'])
eyepacs_class_weight = {i: w for i, w in enumerate(cw)}
span = cw.max()/cw.min()
print(f"\nEyePACS class weight span: {span:.1f}x (expect ~33.7x)")
print("Watch AUC closely if accuracy/QWK looks broken, per Section 7.6. Custom CNN is the")
print("architecture most exposed to this risk, based on its behavior on the pooled set.")

# ================================================================
# STAGE 11: EYEPACS, CUSTOM CNN then EFFICIENTNETB0
# ================================================================

def make_source_gens(preprocess_fn, tr_df, va_df, te_df):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='grade', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=GRADES, color_mode='rgb')
    tr = train_idg.flow_from_dataframe(tr_df, shuffle=True,  seed=SEED, **common)
    va = eval_idg.flow_from_dataframe(va_df,  shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(te_df,  shuffle=False, **common)
    return tr, va, te

def build_custom_cnn(num_classes=5, shape=(224,224,3)):
    return Sequential([
        Input(shape=shape),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'), Dropout(0.5),
        Dense(num_classes,activation='softmax')
    ])

def build_pretrained(base_class, num_classes=5, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(),
                         Dense(256,activation='relu'), Dropout(0.3),
                         Dense(num_classes,activation='softmax')])
    return model, base

stage11_results = []

# ---------- Custom CNN, EyePACS ----------
tag = "ss_custom_eyepacs_dr"
tr, va, te = make_source_gens(None, e_tr, e_va, e_te)
model = build_custom_cnn()
model.compile(Adam(CUSTOM_LR), 'categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
cbs = [EarlyStopping(monitor=MONITOR, patience=EARLYSTOP_PAT, restore_best_weights=True),
       ModelCheckpoint(f'/kaggle/working/{tag}.keras', monitor=MONITOR, save_best_only=True),
       CSVLogger(f'/kaggle/working/{tag}_log.csv', append=False)]
print(f"\n===== Custom CNN, EyePACS (weight span {span:.1f}x) =====")
model.fit(tr, validation_data=va, epochs=60, class_weight=eyepacs_class_weight, callbacks=cbs, verbose=1)
result = model.evaluate(te, verbose=0)
print(f"\n{tag} TEST: loss={result[0]:.4f} accuracy={result[1]:.4f} auc={result[2]:.4f}")
if result[2] > 0.70 and result[1] < 0.45:
    print("  ^ WATCH: AUC comparatively healthy but accuracy weak, this is the collapse signature.")
stage11_results.append({'arch':'custom','source':'eyepacs','loss':result[0],'accuracy':result[1],'auc':result[2]})
pd.DataFrame(stage11_results).to_csv('/kaggle/working/dr_stage11_eyepacs_custom.csv', index=False)
print("Checkpointed after Custom CNN.")
del model
tf.keras.backend.clear_session()

# ---------- EfficientNetB0, EyePACS ----------
tag_p1 = "ss_eff_eyepacs_dr_phase1"
tag_p2 = "ss_eff_eyepacs_dr"
tr, va, te = make_source_gens(eff_pre, e_tr, e_va, e_te)
model, base = build_pretrained(EfficientNetB0)

base.trainable = False
model.compile(Adam(PHASE1_LR), 'categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
print(f"\n===== eff, eyepacs: PHASE 1 (head only, {PHASE1_EPOCHS} epochs) =====")
model.fit(tr, validation_data=va, epochs=PHASE1_EPOCHS, class_weight=eyepacs_class_weight,
          callbacks=[ModelCheckpoint(f'/kaggle/working/{tag_p1}.keras', monitor=MONITOR, save_best_only=True),
                     CSVLogger(f'/kaggle/working/{tag_p1}_log.csv', append=False)], verbose=1)
print("eff, eyepacs Phase 1 saved.")

base.trainable = True
model.compile(Adam(PHASE2_LR), 'categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
print(f"\n===== eff, eyepacs: PHASE 2 (full fine-tune, up to 60 epochs) =====")
model.fit(tr, validation_data=va, epochs=60, class_weight=eyepacs_class_weight,
          callbacks=[EarlyStopping(monitor=MONITOR, patience=EARLYSTOP_PAT, restore_best_weights=True),
                     ModelCheckpoint(f'/kaggle/working/{tag_p2}.keras', monitor=MONITOR, save_best_only=True),
                     CSVLogger(f'/kaggle/working/{tag_p2}_log.csv', append=False)], verbose=1)
result = model.evaluate(te, verbose=0)
print(f"\n{tag_p2} TEST: loss={result[0]:.4f} accuracy={result[1]:.4f} auc={result[2]:.4f}")
print("eff, eyepacs Phase 2 saved.")
stage11_results.append({'arch':'eff','source':'eyepacs','loss':result[0],'accuracy':result[1],'auc':result[2]})
pd.DataFrame(stage11_results).to_csv('/kaggle/working/dr_stage11_eyepacs_custom_eff.csv', index=False)
print("Checkpointed after EfficientNetB0.")

print("\n===== EYEPACS, Custom CNN + EfficientNetB0, COMPLETE =====")
print(pd.DataFrame(stage11_results).to_string(index=False))
print("\n===== EYEPACS SOURCE COMPLETE: all four architectures done =====")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 40.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-20 09:37:14.169370: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787218634.191737      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787218634.198978      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787218634.216831      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787218634.216860      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787218634.216862      22 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
EyePACS subsampled: 3662 images, 1831 patients
EyePACS patient-leakage check: PASS

EyePACS class weight span: 33.7x (expect ~33.7x)
Watch AUC closely if accuracy/QWK looks broken, per Section 7.6. Custom CNN is the
architecture most exposed to this risk, based on its behavior on the pooled set.
Found 2562 validated image filenames belonging to 5 classes.
Found 550 validated image filenames belonging to 5 classes.
Found 550 validated image filenames belonging to 5 classes.


I0000 00:00:1787218647.829714      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787218647.832574      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



===== Custom CNN, EyePACS (weight span 33.7x) =====
Epoch 1/60


...

81/81 [==============================] - 86s 953ms/step - loss: 1.8120 - accuracy: 0.2205 - auc: 0.5294 - val_loss: 1.6005 - val_accuracy: 0.1473 - val_auc: 0.5476
Epoch 2/60
81/81 [==============================] - 65s 808ms/step - loss: 1.7438 - accuracy: 0.1983 - auc: 0.5214 - val_loss: 1.4841 - val_accuracy: 0.0691 - val_auc: 0.5582
Epoch 3/60
81/81 [==============================] - 65s 804ms/step - loss: 1.6699 - accuracy: 0.2279 - auc: 0.5783 - val_loss: 2.4110 - val_accuracy: 0.0691 - val_auc: 0.5839
Epoch 4/60
81/81 [==============================] - 65s 803ms/step - loss: 1.6577 - accuracy: 0.2108 - auc: 0.5575 - val_loss: 1.9162 - val_accuracy: 0.0691 - val_auc: 0.5648
Epoch 5/60
81/81 [==============================] - 65s 807ms/step - loss: 1.6344 - accuracy: 0.2533 - auc: 0.5997 - val_loss: 1.6950 - val_accuracy: 0.0982 - val_auc: 0.4178
Epoch 6/60
81/81 [==============================] - 65s 802ms/step - loss: 1.6418 - accuracy: 0.1784 - auc: 0.5348 - val_loss: 1.8585 - 

E0000 00:00:1787219209.292926      22 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


81/81 [==============================] - 60s 662ms/step - loss: 1.5787 - accuracy: 0.3017 - auc: 0.6525 - val_loss: 1.3790 - val_accuracy: 0.3473 - val_auc: 0.7354
Epoch 2/10
81/81 [==============================] - 51s 624ms/step - loss: 1.3271 - accuracy: 0.3716 - auc: 0.7313 - val_loss: 1.3361 - val_accuracy: 0.2945 - val_auc: 0.7202
Epoch 3/10
81/81 [==============================] - 50s 621ms/step - loss: 1.2605 - accuracy: 0.3841 - auc: 0.7401 - val_loss: 1.5707 - val_accuracy: 0.2309 - val_auc: 0.6275
Epoch 4/10
81/81 [==============================] - 51s 629ms/step - loss: 1.2143 - accuracy: 0.3708 - auc: 0.7414 - val_loss: 1.0711 - val_accuracy: 0.6709 - val_auc: 0.8535
Epoch 5/10
81/81 [==============================] - 51s 625ms/step - loss: 1.1630 - accuracy: 0.4286 - auc: 0.7701 - val_loss: 1.1976 - val_accuracy: 0.6091 - val_auc: 0.8194
Epoch 6/10
81/81 [==============================] - 50s 621ms/step - loss: 1.1092 - accuracy: 0.4477 - auc: 0.7840 - val_loss: 1.1028 - 

E0000 00:00:1787219735.699675      22 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


81/81 [==============================] - 112s 844ms/step - loss: 1.5657 - accuracy: 0.3259 - auc: 0.6796 - val_loss: 1.1339 - val_accuracy: 0.6127 - val_auc: 0.8404
Epoch 2/60
81/81 [==============================] - 64s 792ms/step - loss: 1.4906 - accuracy: 0.3228 - auc: 0.6735 - val_loss: 1.1166 - val_accuracy: 0.5909 - val_auc: 0.8409
Epoch 3/60
81/81 [==============================] - 64s 792ms/step - loss: 1.4643 - accuracy: 0.3220 - auc: 0.6808 - val_loss: 1.1216 - val_accuracy: 0.5709 - val_auc: 0.8388
Epoch 4/60
81/81 [==============================] - 64s 793ms/step - loss: 1.3538 - accuracy: 0.3388 - auc: 0.6905 - val_loss: 1.1389 - val_accuracy: 0.5491 - val_auc: 0.8306
Epoch 5/60
81/81 [==============================] - 65s 797ms/step - loss: 1.3445 - accuracy: 0.3365 - auc: 0.6867 - val_loss: 1.1682 - val_accuracy: 0.5182 - val_auc: 0.8170
Epoch 6/60
81/81 [==============================] - 65s 799ms/step - loss: 1.3198 - accuracy: 0.3310 - auc: 0.6902 - val_loss: 1.1825 -